In [1]:
import pickle
import os

import sys
from pathlib import Path

project_root = Path(r"C:\Users\zcohe\Jmod\JMod")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


project_root = Path(r"C:\Users\zcohe\Jmod\JMod")
sys.path.insert(0, str(project_root))
import src.run_jmod
import src.config

import numpy as np

pickle_path = r"C:\Users\zcohe\Jmod\JMod_Profiling\Output\Line_Profiler_MS1_Cor_Channels\MS1_cor_inouts_final.pkl"
with open(pickle_path, "rb") as f:
    pickle_dic = pickle.load(f)
print(pickle_dic.keys())
group_p_corrs_out = pickle_dic.get("group_p_corrs")
group_ms1_traces_out = pickle_dic.get("group_ms1_traces")
group_ms2_traces_out = pickle_dic.get("group_ms2_traces")
group_iso_ratios_out = pickle_dic.get("group_iso_ratios")
group_keys_out = pickle_dic.get("group_keys")
group_fitted_out = pickle_dic.get("group_fitted")
DIAspectra = pickle_dic.get("DIAspectra")
fdc = pickle_dic.get("fdc")
dc = pickle_dic.get("dc")
mz_ppm = pickle_dic.get("mz_ppm")
rt_tol = pickle_dic.get("rt_tol")
mass_tag = pickle_dic.get("mass_tag")
timeplex = pickle_dic.get("timeplex")

00:00:17 - WARNING - From c:\Users\zcohe\miniconda3\envs\jmod\Lib\site-packages\keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.

dict_keys(['DIAspectra', 'fdc', 'dc', 'mz_ppm', 'rt_tol', 'mass_tag', 'timeplex', 'group_p_corrs', 'group_ms1_traces', 'group_ms2_traces', 'group_iso_ratios', 'group_keys', 'group_fitted'])


In [2]:
from pyteomics import mass
class massTag():
    
    def __init__(self,rules,base_mass,delta,channel_names, name, compositions=None):
        
        self.rules = rules
        
        self.mass = base_mass
        
        self.delta = delta
        
        self.n_channels = len(channel_names)
        
        self.channel_names = channel_names
        
        if type(delta)!= list and len(delta)<2:
            self.channel_masses =(np.arange(self.n_channels)*delta)+base_mass
        else:
            assert len(delta)==len(self.channel_names), "Channel names and deltas do not match"
            self.channel_masses =(np.ones(self.n_channels)*delta)+base_mass
        self.name = name
        
        self.mass_dict = {self.name+"-"+str(i):j for i,j in zip(self.channel_names,self.channel_masses)}
    
    
        if compositions is not None:
            self.channel_comp = {i:compositions[i] for i in self.channel_names}
        else: 
            self.channel_comp=None
            
    def __repr__(self):
        return("\n".join([
                           "Mass Tag",
                          # f"{self.n_channels} Channels",
                          F"TagName: {self.name}",
                          f"Base Mass: {self.mass}",
                          f"MassDelta(s): {self.delta}",
                          f"ChannelNames: {self.channel_names}",
                          f"ChannelMasses: {self.channel_masses}"]))
    
    def __getitem__(self,item):
        return getattr(self,item)

composition_dict = {
        "0": {
            "C": 18,
            "H": 16,
            "N": 2,
            "O": 3
        },
        "4": {
            "C": 16,
            "H": 16,
            "N": 2,
            "O": 2,
            "C[13]": 2,
            "O[18]": 1
        },
        "8": {
            "C": 10,
            "H": 16,
            "N": 2,
            "O": 3,
            "C[13]": 8
        },
        "12": {
            "C": 7,
            "H": 16,
            "N": 1,
            "O": 3,
            "C[13]": 11,
            "N[15]": 1
        },
        "16": {
            "C": 6,
            "H": 16,
            "N": 0,
            "O": 2,
            "C[13]": 12,
            "N[15]": 2,
            "O[18]": 1
        }
    }
compositions = {channel: mass.Composition(comp_dict) for channel, comp_dict in composition_dict.items()}

PSMtag_5plex = massTag(rules="nK", base_mass=308.1160923903, delta=[0.0, 4.01095605604, 8.0268387024, 12.0339381092, 16.03857422084], channel_names=['0', '4', '8', '12', '16'], name="PSMtag_5plex", compositions=compositions)
print(PSMtag_5plex.channel_comp)

{'0': Composition({'C': 18, 'H': 16, 'N': 2, 'O': 3}), '4': Composition({'C': 16, 'H': 16, 'N': 2, 'O': 2, 'C[13]': 2, 'O[18]': 1}), '8': Composition({'C': 10, 'H': 16, 'N': 2, 'O': 3, 'C[13]': 8}), '12': Composition({'C': 7, 'H': 16, 'N': 1, 'O': 3, 'C[13]': 11, 'N[15]': 1}), '16': Composition({'C': 6, 'H': 16, 'O': 2, 'C[13]': 12, 'N[15]': 2, 'O[18]': 1})}


In [3]:
# print(type(fdc))
# print(len(fdc))
# print(fdc.shape)
# print(fdc.columns)
#print(list(dc.iloc[0, :]))

import re

def get_other_channels(prec,mz,tag):
    print(f"prec: {prec}")
    ### want to return m/z and seqs for all channels including this one
    
    ## identify what channel the current prec is in
    channels = re.findall(f"({tag.name}-\d+)",prec[0])
    num_tags = len(channels)
    assert len(set(channels))==1, f"{channels}"
    channel = channels[0]
    assert channel in tag.mass_dict
    channel_dict = {i:[] for i in tag.mass_dict}
    
    for c in channel_dict:
        if c==channel:
            channel_dict[channel] = [prec[0],mz]
        else:
            c_seq = re.sub(channel,c,prec[0])
            c_mz = mz + (num_tags*(tag.mass_dict[c]-tag.mass_dict[channel])/prec[1])
            channel_dict[c] = [c_seq,c_mz]
            
    return channel_dict

fdc_group = fdc.groupby(["untag_seq","z"])
print(type(fdc_group))
# print(fdc_group.first())
for i, key in enumerate(list(fdc_group.groups)):
    if i == 1930:
        print(key)
        tag_group = fdc_group.get_group(key)
        #print(tag_group)
        print(tag_group["Ms1_spec_id"])
        prec_mzs = tag_group["mz"]
        print(prec_mzs)
        prec_seqs = tag_group["seq"]
        print(prec_seqs)
        prec_z = key[1]
        print(prec_z)
        print(tag_group["coeff"])
        largest_id = np.argmax(tag_group["coeff"])
        print(largest_id)
        print("\n")
        channel_dict = get_other_channels((prec_seqs.iloc[largest_id],prec_z), prec_mzs.iloc[largest_id], PSMtag_5plex)
        print(channel_dict)

        largest_coeff_scans = list(fdc["Ms1_spec_id"][np.logical_and(fdc["untag_seq"]==key[0],
                                                                                           fdc["z"]==key[1])])
        largest_coeff_scans2 = list(tag_group["Ms1_spec_id"])
        print(largest_coeff_scans)
        print(largest_coeff_scans2)
        break


<class 'pandas.core.groupby.generic.DataFrameGroupBy'>
('GELASYDMR', 2.0)
3562     12823
9445     12801
19099    12823
21462    12845
Name: Ms1_spec_id, dtype: int32
3562     679.314331
9445     683.320190
19099    677.306335
21462    675.300842
Name: mz, dtype: float32
3562      G(PSMtag_5plex-8)ELASYDMR
9445     G(PSMtag_5plex-16)ELASYDMR
19099     G(PSMtag_5plex-4)ELASYDMR
21462     G(PSMtag_5plex-0)ELASYDMR
Name: seq, dtype: object
2.0
3562      1414.336060
9445      2877.859619
19099     9610.305664
21462    18793.482422
Name: coeff, dtype: float32
3


prec: ('G(PSMtag_5plex-0)ELASYDMR', 2.0)
{'PSMtag_5plex-0': ['G(PSMtag_5plex-0)ELASYDMR', 675.30084], 'PSMtag_5plex-4': ['G(PSMtag_5plex-4)ELASYDMR', 677.3063203131762], 'PSMtag_5plex-8': ['G(PSMtag_5plex-8)ELASYDMR', 679.3142616363563], 'PSMtag_5plex-12': ['G(PSMtag_5plex-12)ELASYDMR', 681.3178113397562], 'PSMtag_5plex-16': ['G(PSMtag_5plex-16)ELASYDMR', 683.3201293955763]}
[12823, 12801, 12823, 12845]
[12823, 12801, 12823, 12845]


In [13]:
from scipy.interpolate import interp1d
from brainpy import isotopic_variants
from functools import reduce
from pyteomics import mass
unimods = mass.Unimod()


def closest_ms1spec(ms2rt,ms1rt):
    """
    Parameters
    ----------
    ms2rt : float
        Single rt from ms2 spectrum in question.
    ms1rt : float
        numpy array of RTs for all MS2 spectra.

    Returns
    -------
    closest_idx : int
        Index of cosest MS1 spectrum in RT space.

    """
    closest_idx = np.argmin(np.abs(ms1rt-ms2rt))
    return closest_idx
def get_seqs_and_mzs(fdc_group, timeplex, tag, key):
    tag_group = fdc_group.get_group(key)
    prec_mzs = tag_group["mz"]
    prec_seqs = tag_group["seq"]
    prec_z = key[1]
    if timeplex:
        time_channel = key[2]
    else:
        time_channel = None
    largest_id = np.argmax(tag_group["coeff"])
    top_ms1_spec_idx = list(tag_group["Ms1_spec_id"])[largest_id]
    prec_rt = list(tag_group["rt"])[largest_id]
    
    ### search for all channels always:
    channel_dict = get_other_channels((prec_seqs.iloc[largest_id],prec_z), prec_mzs.iloc[largest_id], tag)
    prec_seqs,prec_mzs = tuple(zip(*channel_dict.values()))

    largest_coeff_scans = list(tag_group["Ms1_spec_id"])

    return prec_seqs, prec_mzs, prec_z, prec_rt, top_ms1_spec_idx, largest_coeff_scans, time_channel

def get_other_channels(prec,mz,tag):
    ### want to return m/z and seqs for all channels including this one
    
    ## identify what channel the current prec is in
    channels = re.findall(f"({tag.name}-\d+)",prec[0])
    num_tags = len(channels)
    assert len(set(channels))==1, f"{channels}"
    channel = channels[0]
    assert channel in tag.mass_dict
    channel_dict = {i:[] for i in tag.mass_dict}
    
    for c in channel_dict:
        if c==channel:
            channel_dict[channel] = [prec[0],mz]
        else:
            c_seq = re.sub(channel,c,prec[0])
            c_mz = mz + (num_tags*(tag.mass_dict[c]-tag.mass_dict[channel])/prec[1])
            channel_dict[c] = [c_seq,c_mz]
            
    return channel_dict

def minmax_spec_window(largest_coeff_scans, ms1_spec_idxs, ms1_spectra, all_spectra, window_half_width): 
    ## max and min of this list
    max_scan, min_scan = max(largest_coeff_scans), min(largest_coeff_scans)
    ms1_list_idx_min = list(ms1_spec_idxs).index(min_scan)
    ms1_list_idx_max = list(ms1_spec_idxs).index(max_scan)
    scans_each_side = np.array(ms1_spec_idxs)[np.arange(max(0,ms1_list_idx_min-window_half_width),min(len(ms1_spectra),ms1_list_idx_max+window_half_width+1))]
    all_scans = list(scans_each_side)

    spectra_subset = [all_spectra.get_by_idx(idx) for idx in all_scans]
    return all_scans, spectra_subset


def get_isotopes_and_vals(prec_seq, prec_z, num_iso, tag, all_scans, prec_mz, mz_ppm, spectra_subset, interp_func):
    ms1_vals = get_precursor_trace(prec_mz, mz_ppm, spectra_subset)
    isotopes = compute_isotopes(prec_seq, prec_mz, prec_z, num_iso, tag)
    prec_isotope_traces = get_isotope_traces(isotopes, mz_ppm, spectra_subset)
    all_ms1_vals, all_ms2_vals, all_iso_vals = unnamed_function(all_scans, prec_isotope_traces, interp_func, ms1_vals)

    return all_ms1_vals, all_ms2_vals, all_iso_vals, isotopes, interp_func

def build_ms2_interpolator(ms2_vals):
    return interp1d(list(ms2_vals.keys()), np.array(list(ms2_vals.values())), bounds_error=False)   

def get_precursor_trace(prec_mz, mz_ppm, spectra_subset):
    return {spec.scan_num:get_trace_int(spec, prec_mz,rtol=mz_ppm) for spec in spectra_subset}

def get_isotope_traces(isotopes, mz_ppm, spectra_subset):
    prec_isotope_traces=[]
    ## note: we have collected similar values for previous channel if the isotopic envelopes are overlapping. 
    ### However, in cases like diethlyation, isoptopes can differ by > 10 ppm #!!!Maybe investigate wider ppm tol for these cases?
    for isotope in isotopes[1:]:# we already have the monoisotopic trace
        iso_trace = {spec.scan_num:get_trace_int(spec, isotope.mz,rtol=mz_ppm) for spec in spectra_subset}
        prec_isotope_traces.append(iso_trace)
    
    return prec_isotope_traces
           
def compute_isotopes(prec_seq, prec_mz, prec_z, num_iso, tag):
    isotopes = precursor_isotopes(prec_seq,prec_z,num_iso)

    delta_mz = 0
    if tag.name in prec_seq:
        delta_mz = prec_mz-isotopes[0].mz
    for i in isotopes:
        i.mz+=delta_mz

    return isotopes

def unnamed_function(all_scans, prec_isotope_traces, interp_func, ms1_vals):
    all_ms1_vals = {i:min_int for i in all_scans}
    all_ms2_vals = {i:min_int for i in all_scans}
    all_iso_vals = [{i:min_int for i in all_scans} for _ in range(len(prec_isotope_traces))]
    
    for scan,c in zip(all_scans,interp_func(all_scans)):
        if scan in ms1_vals:
            all_ms1_vals[scan] = ms1_vals[scan]
            all_ms2_vals[scan] = c#f(scan)
        for iso_idx in range(len(prec_isotope_traces)):
            if scan in prec_isotope_traces[iso_idx]:
                all_iso_vals[iso_idx][scan] = prec_isotope_traces[iso_idx][scan]

    return all_ms1_vals, all_ms2_vals, all_iso_vals


def get_trace_int(spec,mz,atol=0,rtol=0,base=min_int):
    ## speed up of above
    order_idx = np.searchsorted(spec.mz, mz)
    
    # Handle edge cases for indices at the bounds
    if order_idx == 0:
        closest_idx = 0
        mz_diff = spec.mz[0]-mz
    elif order_idx == len(spec.mz):
        closest_idx = len(spec.mz) - 1
        mz_diff = mz-spec.mz[-1]
    else:
        # Compare the closest values on both sides of the searchsorted index
        left_idx = order_idx - 1
        right_idx = order_idx
        
        # Find the closest value between the two neighboring indices
        left_diff = abs(spec.mz[left_idx] - mz)
        right_diff = abs(spec.mz[right_idx] - mz)
        if left_diff < right_diff:
            closest_idx = left_idx
            mz_diff = left_diff
        else:
            closest_idx = right_idx
            mz_diff = right_diff
    
#    mz_diff = abs(spec.mz[closest_idx] - mz)
    if mz_diff <= mz * rtol:  # Use the relative tolerance condition
        return spec.intens[closest_idx]

    return base


def precursor_isotopes(sequence,charge,n_isotopes=2):
    sequence = re.sub("Decoy_","",sequence)
    #split_seq = split_peptide(sequence)
    split_seq = parse_peptide(sequence)
    
    seq_comp = get_seq_comp(split_seq, "M")
    
    if PSMtag_5plex:
        tags = [t for aa in split_seq for t in re.findall(f"\(({PSMtag_5plex.name}.*?)\)",aa)]
        if PSMtag_5plex.channel_comp is not None and len(tags)>0:
                tag_comp = reduce(lambda x, y: x + y, [PSMtag_5plex.channel_comp[re.findall(f"{PSMtag_5plex.name}-(\d+)",t)[0]] for t in tags])
                seq_comp+=tag_comp
            
    
    isotopes = isotopic_variants(seq_comp,
                                 npeaks=n_isotopes,
                                 charge = int(charge))
    
    return isotopes

def parse_peptide(seq):
    close_d = {"[": "]", "(": ")"}
    open_set = set(close_d.keys())
    close_set = set(close_d.values())
    
    new_seq = []
    current = ""
    s_idx = 0

    while s_idx < len(seq):
        s = seq[s_idx]

        if s in open_set:
            # Begin collecting the bracketed modification
            opener = s
            closer = close_d[opener]
            mod = s
            stack = [closer]
            s_idx += 1

            while s_idx < len(seq) and stack:
                c = seq[s_idx]
                mod += c

                if c in open_set:
                    stack.append(close_d[c])
                elif c in close_set:
                    if stack and c == stack[-1]:
                        stack.pop()
                s_idx += 1

            current += mod  # Append full modification to current letter

        elif s.isalpha():
            if current:
                new_seq.append(current)
            current = s
            s_idx += 1

        else:
            # If somehow an unexpected char, just add it
            current += s
            s_idx += 1

    if current:
        new_seq.append(current)

    return new_seq

def get_seq_comp(split_seq,ion_type):
    
    stripped_seq = "".join([i[0] for i in split_seq]) ## assumes AA comes first before mods
    
    mods = [int(j) for i in split_seq for j in re.findall("\([A-z]+\:(\d+)\)",i) if len(i)>1]
    # tags = [t for aa in split_seq for t in re.findall("(\(.*?\))",aa)]
    seq_comp = mass.Composition(sequence=stripped_seq,ion_type=ion_type)
    for unimod_idx in mods:
        seq_comp += unimods.by_id(unimod_idx)["composition"]
    return seq_comp


def get_ms2_vals(variable_to_test, prec_seq, prec_z, prec_rt, time_channel, timeplex, grouped_decoy_coeffs, lesser_features_present, greater_features_present, ms2_rt, rt_tol, prec_mz, bottom_of_window, top_of_window, ms2_spec_idxs):
    ## keep decoys mathching to the correct MS1
    offset = 0 if "Decoy" in prec_seq else 0
    
    if timeplex:
        channel_key = (prec_seq,prec_z,time_channel)
    else:
        channel_key = (prec_seq,prec_z)

    #print(f"channel_key, {channel_key}")
    
    ## create dummy 
    ms2_vals = {0:0}
    
    if channel_key in grouped_decoy_coeffs.groups:
        #print("channel key in gdc")

        new_data= grouped_decoy_coeffs.get_group(channel_key).copy()
        new_data2= grouped_decoy_coeffs.get_group(channel_key).copy()
        # print(f"new_data_shape: {new_data.shape}")
        #print(f"new_data_columns: {new_data.columns}")
        
        ### rank order the coeffs in terms of goodness of fit
        new_data.loc[:,"rank_score"] = np.sum([np.argsort(-new_data.loc[:,i]).argsort() for i in lesser_features_present],0)               
        new_data.loc[:,"rank_score"] += np.sum([np.argsort(new_data[i]).argsort() for i in greater_features_present],0)



        try:
            new_data2.loc[:,"rank_score"] = new_data2.loc[:, variable_to_test]
        except:
            print(f"key error: {variable_to_test}")
            return False, False, False


        # print(f"new_data_shape: {new_data.shape}")
        # print(f"new_data_columns: {new_data.columns}")
        
        highest_ranked_spec = new_data.Ms1_spec_id.iloc[np.argmax(new_data.rank_score)]
        highest_ranked_spec_coeff = new_data2.Ms1_spec_id.iloc[np.argmax(new_data2.rank_score)]

        coeff_matches_ranked = True
        global hyperscore_false, hyperscore_all
        if not highest_ranked_spec == highest_ranked_spec_coeff:
            coeff_matches_ranked = False
            hyperscore_false += 1
        hyperscore_all += 1
                                        
        ms2_rt_bool = np.abs(ms2_rt-prec_rt)<rt_tol
        prec_rt = new_data.rt.iloc[np.argmax(new_data.coeff)]
        ms2_window_bool = np.logical_and(prec_mz+offset>bottom_of_window,prec_mz+offset<top_of_window)
        
        min_rt = np.minimum(prec_rt-rt_tol,np.min(new_data.rt)*.99)
        max_rt = np.maximum(prec_rt+rt_tol,np.max(new_data.rt)*1.01)
        ms2_rt_bool = np.logical_and(ms2_rt>=min_rt,ms2_rt<=max_rt)
        
        ms2_bool = np.logical_and(ms2_window_bool,ms2_rt_bool)
        possible_ms2_scans = ms2_spec_idxs[ms2_bool]
        ms2_vals = {i:min_int for i in possible_ms2_scans}
    
        for scan,c in zip(new_data["spec_id"],new_data["coeff"]):
            ms2_vals[scan]=c
    else:
        highest_ranked_spec = None
        highest_ranked_spec_coeff = None
        coeff_matches_ranked = None

    #print("\n")
    return ms2_vals, highest_ranked_spec, highest_ranked_spec_coeff, channel_key, coeff_matches_ranked


In [17]:
def ms1_cor_channels(all_spectra,filtered_decoy_coeffs,decoy_coeffs,mz_ppm,rt_tol, desired_iter_val, desired_iter_val_2, variable_to_test, tag=None,timeplex=False):
    global hyperscore_false, hyperscore_all
    hyperscore_false = 0
    hyperscore_all = 0
    decoy_coeffs["untag_seq"] = [re.sub(f"(\({tag.name}-\d+\))?","",peptide) for peptide in decoy_coeffs["seq"]]
    decoy_coeffs["untag_prec"] = ["_".join([i[0],str(int(i[1]))]) for i in zip(decoy_coeffs["untag_seq"],decoy_coeffs["z"])]
    
    if "med_frag_error" not in decoy_coeffs.columns:
        frag_errors = [mf.unstring_floats(mz) if mz==mz else [] for mz in decoy_coeffs.frag_errors]
        median  = np.median(np.concatenate([i for i in frag_errors]))
        decoy_coeffs["med_frag_error"] = [np.median(np.abs(median-i)) for i in frag_errors]
    
    if "abs_rt_error" not in decoy_coeffs.columns:
        decoy_coeffs["abs_rt_error"] = np.abs(decoy_coeffs.rt_error)
    
    if "abs_mz_error" not in decoy_coeffs.columns:
        decoy_coeffs["abs_mz_error"] = np.abs(decoy_coeffs.mz_error)
        
    
    ## features where bigger is better 
    greater_features =["hyperscore","frag_cosines_p","frag_cosines_p","manhattan_distances","coeff","frac_lib_int"]
    greater_features_present = [i for i in greater_features if i in decoy_coeffs.columns and np.ptp(decoy_coeffs[i][~np.isnan(decoy_coeffs[i])])>0]
    
    lesser_features =["scribe_scores","gof_stats","max_matched_residuals","med_frag_error","abs_mz_error","abs_rt_error"]
    lesser_features_present = [i for i in lesser_features if i in decoy_coeffs.columns and np.ptp(decoy_coeffs[i][~np.isnan(decoy_coeffs[i])])>0]
    
    
    ms1_spectra = all_spectra.ms1scans
    ms2_spectra = all_spectra.ms2scans
    
    ## array of ms1 and ms2 retention time
    ms2_rt = np.array([i.RT for i in ms2_spectra])
    ms1_rt = np.array([i.RT for i in ms1_spectra])
    
    ## array of scan numbers for ms1 and ms2 spectra
    ms1_spec_idxs = np.array([i.scan_num for i in ms1_spectra])
    ms2_spec_idxs = np.array([i.scan_num for i in ms2_spectra])
    
    ## get ms2 info for filtering
    bottom_of_window, top_of_window = np.array([i.ms1window for i in all_spectra.ms2scans]).T
    ms2_rt = np.array([i.RT for i in all_spectra.ms2scans])

    ## list of scan nums of the closest ms1 scan for each ms2 scan
    resp_ms1scans = [ms1_spec_idxs[closest_ms1spec(ms2_rt[i], ms1_rt)] for i in range(len(ms2_rt))]

    ## mapping of ms2 scan nums to ms1 scan nums
    ms2_ms1_scan_map = {spec.scan_num:resp_ms1scans[i] for i,spec in enumerate(all_spectra.ms2scans)}

    
    if timeplex:
        grouped_decoy_coeffs = decoy_coeffs.groupby(["seq","z","time_channel"])
        fdc_group = filtered_decoy_coeffs.groupby(["untag_seq","z","time_channel"])
    else:
        grouped_decoy_coeffs = decoy_coeffs.groupby(["seq","z"])
        fdc_group = filtered_decoy_coeffs.groupby(["untag_seq","z"])

    all_ms1, all_coeff, all_iso, all_group_pearson, all_trace, all_fitted, all_group_keys, all_scans_len = ([] for _ in range(8))

    num_iso = 6
    num_iso_r = 2
    window_half_width = 10
    
    for iter_val, key in enumerate(list(fdc_group.groups)):
        # if iter_val != desired_iter_val:
        #     continue
        if iter_val > 100:
            continue
        prec_seqs, prec_mzs, prec_z, prec_rt, top_ms1_spec_idx, largest_coeff_scans, time_channel = get_seqs_and_mzs(fdc_group, timeplex, tag, key)
        # for item in list(zip(["prec_seqs", "prec_mzs", "prec_z", "prec_rt", "top_ms1_spec_idx", "largest_coeff_scans", "time_channel"],[prec_seqs, prec_mzs, prec_z, prec_rt, top_ms1_spec_idx, largest_coeff_scans, time_channel])):
        #     if type(item[1]) == float or type(item[1]) == int:
        #         print(f"len: 1, {item[0]}: {item[1]}")
        #     elif item[1] == None:
        #         print(f"len: 0, {item[0]}: {item[1]}")
        #     else:
        #         print(f"len: {len(item[1])}, {item[0]}: {item[1]}")
        # print("\n")

        all_scans, spectra_subset = minmax_spec_window(largest_coeff_scans, ms1_spec_idxs, ms1_spectra, all_spectra, window_half_width)
        # for item in list(zip(["all_scans", "spectra_subset"], [all_scans, spectra_subset])):
        #     if type(item[1]) == float or type(item[1]) == int:
        #             print(f"len: 1, {item[0]}: {item[1]}")
        #     elif item[1] == None:
        #         print(f"len: 0, {item[0]}: {item[1]}")
        #     else:
        #         print(f"len: {len(item[1])}, {item[0]}: {item[1]}")
        # print("\n")
        
        ms1_traces, coeff_traces, is_traces, all_pearson, iso_ratios = ([] for _ in range(5))
        obs_ratios, group_iso, group_keys, all_channel_scans, interp_funcs, best_coeff = ([] for _ in range(6))

        for iter_val_2, (prec_mz,prec_seq) in enumerate(zip(prec_mzs,prec_seqs)):
            # if iter_val_2 != desired_iter_val_2:
            #     continue
            

            #print("inputs:")
            strings = ["prec_seq", "prec_z", "prec_rt", "time_channel", "timeplex", "grouped_decoy_coeffs", "lesser_features_present", "greater_features_present", "ms2_rt", "rt_tol", "prec_mz", "bottom_of_window", "top_of_window", "ms2_spec_idxs"]
            not_strings = [prec_seq, prec_z, prec_rt, time_channel, timeplex, grouped_decoy_coeffs, lesser_features_present, greater_features_present, ms2_rt, rt_tol, prec_mz, bottom_of_window, top_of_window, ms2_spec_idxs]
            # for item in list(zip(strings, not_strings)):
            #     if type(item[1]) == float or type(item[1]) == int or type(item[1]) == np.int32 or type(item[1]) == bool or type(item[1]) == np.float64 or type(item[1]) == np.float32:
            #             print(f"type: {type(item[1])}, len: 1, {item[0]}: {item[1]}")
            #     elif item[1] is None:
            #         print(f"type: {type(item[1])}, len: 0, {item[0]}: {item[1]}")
            #     else:
            #         print(f"type: {type(item[1])}, len: {len(item[1])}, {item[0]}: {item[1]}")
            # print("\n")

            ms2_vals, highest_ranked_spec, highest_ranked_spec_coeff, channel_key, coeff_matches_ranked = get_ms2_vals(variable_to_test, prec_seq, prec_z, prec_rt, time_channel, timeplex, grouped_decoy_coeffs, lesser_features_present, greater_features_present, ms2_rt, rt_tol, prec_mz, bottom_of_window, top_of_window, ms2_spec_idxs)
            if [ms2_vals, highest_ranked_spec, channel_key] == [False, False, False]:
                return
            # for item in list(zip(["ms2_vals", "highest_ranked_spec", "channel_key"], [ms2_vals, highest_ranked_spec, channel_key])):
            #     if type(item[1]) == float or type(item[1]) == int or type(item[1]) == np.int32:
            #             print(f"type: {type(item[1])}, len: 1, {item[0]}: {item[1]}")
            #     elif item[1] is None:
            #         print(f"type: {type(item[1])}, len: 0, {item[0]}: {item[1]}")
            #     else:
            #         print(f"type: {type(item[1])}, len: {len(item[1])}, {item[0]}: {item[1]}")
            # print("\n")
            interp_func = build_ms2_interpolator(ms2_vals)
            interp_funcs.append(interp_func)

            all_ms1_vals, all_ms2_vals, all_iso_vals, isotopes, interp_func = get_isotopes_and_vals(prec_seq, prec_z, num_iso, tag, all_scans, prec_mz, mz_ppm, spectra_subset, interp_func)
            group_iso.append(isotopes)

            # if coeff_matches_ranked is False:
            # print(highest_ranked_spec)
            # import matplotlib.pyplot as plt
            # %matplotlib inline
            # plt.figure(figsize=(6,4))
            # plt.scatter(list(all_ms1_vals.keys()), list(all_ms1_vals.values()), color="blue")
            # if highest_ranked_spec is not None:
            #     if not coeff_matches_ranked:
            #         plt.scatter(highest_ranked_spec, all_ms1_vals[highest_ranked_spec], color="orange", label="max_rank")
            #         plt.scatter(highest_ranked_spec_coeff, all_ms1_vals[highest_ranked_spec_coeff], color="green", label="max_ms2_coeff")
            #     else:
            #         plt.scatter(highest_ranked_spec_coeff, all_ms1_vals[highest_ranked_spec_coeff], color="purple", label="both")
            # plt.legend()
            # if coeff_matches_ranked is True:
            #     plt_title = f"True\nfdc_group_index:{iter_val}, channel_index:{iter_val_2}"
            # elif coeff_matches_ranked is False:
            #     plt_title = f"False\nfdc_group_index:{iter_val}, channel_index:{iter_val_2}"
            # elif coeff_matches_ranked is None:
            #     plt_title = f"None\nfdc_group_index:{iter_val}, channel_index:{iter_val_2}"
            # plt.title(plt_title)
            # plt.ylabel("MS1 Coeff")
            # plt.xlabel("Scan Index")
            # plt_title = plt_title.replace("\n", "_").replace(":", "_")
            # plt.savefig(os.path.join(r"C:\Users\zcohe\Jmod\JMod_Profiling\Output\Line_Profiler_MS1_Cor_Channels\MS1_rank_graphs", f"{plt_title}.png"))

    print(f"{variable_to_test} == Rank Score = {hyperscore_all-hyperscore_false}/{hyperscore_all}, {((hyperscore_all-hyperscore_false)/hyperscore_all)*100}%")


In [18]:
min_int = 1e-3
desired_iter_val = 126
desired_iter_val_2 = 4
variable_to_test = "coeff"
ms1_cor_channels(DIAspectra, fdc, dc, mz_ppm, rt_tol, desired_iter_val, desired_iter_val_2, variable_to_test, PSMtag_5plex, timeplex)
# for variable_to_test in ["hyperscore","frag_cosines_p","frag_cosines_p","manhattan_distances","coeff","frac_lib_int"]:
#     ms1_cor_channels(DIAspectra, fdc, dc, mz_ppm, rt_tol, desired_iter_val, desired_iter_val_2, variable_to_test, PSMtag_5plex, timeplex)
# print("\n")
# print("Lesser Features")
# for variable_to_test in ["scribe_scores","gof_stats","max_matched_residuals","med_frag_error","abs_mz_error","abs_rt_error"]:
#     ms1_cor_channels(DIAspectra, fdc, dc, mz_ppm, rt_tol, desired_iter_val, desired_iter_val_2, variable_to_test, PSMtag_5plex, timeplex)



coeff == Rank Score = 317/414, 76.57004830917874%
